In [8]:

# Experiment 5: Token Economics - Cost and Rate-Limit Budget
# Feature: Ask the Course Assistant for 1,000 Daily Users

import pandas as pd
import math

# 1. Sample requests
SYSTEM = "You are a course assistant for Generative AI. Answer briefly and cite the course handout."

CONTEXT = (
    "Module III covers embeddings, chunking, MongoDB, "
    "Chroma and RAG evaluation with Hit Rate@k and MRR. "
) * 8

questions = [
    "What is RAG?",
    "Explain MRR with an example.",
    "రాగ్ అంటే ఏమిటి?",
    "आरएजी क्या है?"
]

# Local token estimation (no API required)
def estimate_tokens(text):
    return math.ceil(len(text) / 4)

rows = []

for q in questions:
    prompt = SYSTEM + "\n" + CONTEXT + "\nQuestion: " + q

    input_tokens = estimate_tokens(prompt)

    # Estimated answer length: 100 tokens
    output_tokens = 100

    rows.append({
        "question": q,
        "input_tokens_estimated": input_tokens,
        "output_tokens_estimated": output_tokens
    })

sample = pd.DataFrame(rows)

print("SAMPLE TOKEN COUNTS")
print(sample.to_string(index=False))


# 2. Budget assumptions

USERS_PER_DAY = 1000
QUESTIONS_PER_USER = 5
PEAK_HOUR_SHARE = 0.20

PRICE_IN = 0.10
PRICE_OUT = 0.40

RPM_LIMIT = 15
RPD_LIMIT = 1000

CONTEXT_LIMIT = 1000000

avg_in = sample["input_tokens_estimated"].mean()
avg_out = sample["output_tokens_estimated"].mean()

req_day = USERS_PER_DAY * QUESTIONS_PER_USER

cost_req = (
    avg_in * PRICE_IN + avg_out * PRICE_OUT
) / 1e6

peak_rpm = req_day * PEAK_HOUR_SHARE / 60

budget = {
    "Average input tokens/request": round(avg_in),
    "Average output tokens/request": round(avg_out),
    "Requests/day": req_day,
    "Tokens/day": round(req_day * (avg_in + avg_out)),
    "Cost/request (USD)": round(cost_req, 6),
    "Cost/day (USD)": round(cost_req * req_day, 2),
    "Cost/month (USD)": round(cost_req * req_day * 30, 2),
    "Cost/month (INR @ 84)": round(cost_req * req_day * 30 * 84),
    "Peak requests/minute": round(peak_rpm, 1),
    "Fits free-tier RPM?": peak_rpm <= RPM_LIMIT,
    "Fits free-tier RPD?": req_day <= RPD_LIMIT,
    "Context used/request": f"{avg_in / CONTEXT_LIMIT:.4%}"
}

print("\nCOST BUDGET NOTE")

for k, v in budget.items():
    print(f"{k:30s}: {v}")

print("\nDECISION:")
if peak_rpm > RPM_LIMIT or req_day > RPD_LIMIT:
    print("Add caching, a paid tier, or a request queue before launch.")
else:
    print("The estimated traffic fits the assumed limits.")

print("\nExperiment completed successfully!")
print("Note: Token counts are estimates, not actual Gemini measurements.")


SAMPLE TOKEN COUNTS
                    question  input_tokens_estimated  output_tokens_estimated
                What is RAG?                     229                      100
Explain MRR with an example.                     233                      100
            రాగ్ అంటే ఏమిటి?                     230                      100
              आरएजी क्या है?                     229                      100

COST BUDGET NOTE
Average input tokens/request  : 230
Average output tokens/request : 100
Requests/day                  : 5000
Tokens/day                    : 1651250
Cost/request (USD)            : 6.3e-05
Cost/day (USD)                : 0.32
Cost/month (USD)              : 9.45
Cost/month (INR @ 84)         : 794
Peak requests/minute          : 16.7
Fits free-tier RPM?           : False
Fits free-tier RPD?           : False
Context used/request          : 0.0230%

DECISION:
Add caching, a paid tier, or a request queue before launch.

Experiment completed successfully!
Note: Token c